# RACE RC & Quiz Generation - Final GPU Training (Fixed)
This notebook mirrors `model_a_train.py` for full-dataset training on Google Colab T4 GPU.

In [14]:
# 1. Setup & Installation
!pip install cuml-cu12 cudf-cu12 --extra-index-url https://pypi.nvidia.com
!pip install gensim joblib pandas numpy scikit-learn

import os, joblib, time
import numpy as np
import pandas as pd
import cupy as cp
from google.colab import drive
drive.mount('/content/drive')

# Paths
PROJECT_DIR = '/content/drive/MyDrive/race_rc_project'
DATA_DIR    = f'{PROJECT_DIR}/data/processed'
MODEL_DIR_A = f'{PROJECT_DIR}/models/model_a/traditional'
os.makedirs(MODEL_DIR_A, exist_ok=True)

# Load Data
print("Loading features...")
train_data = joblib.load(f'{DATA_DIR}/train_features.pkl')
val_data   = joblib.load(f'{DATA_DIR}/val_features.pkl')

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val     = val_data['X'], val_data['y']
print(f"Loaded {len(X_train)} training rows.")

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading features...
Loaded 40000 training rows.


## 2. Feature Scaling


In [15]:
from sklearn.preprocessing import StandardScaler

print("Fitting StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

joblib.dump(scaler, f'{MODEL_DIR_A}/scaler.pkl')
print("Scaler saved.")

Fitting StandardScaler...
Scaler saved.


## 3. Supervised Models


In [16]:
from cuml.svm import LinearSVC as cuSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import LinearSVC # Sklearn version for ensemble compatibility

# 3a. Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, f'{MODEL_DIR_A}/lr.pkl')

# 3b. SVM (cuML for speed, saved as standalone)
print("Training cuML SVM for standalone use...")
X_gpu = cp.asarray(X_train_scaled.astype('float32'))
y_gpu = cp.asarray(y_train.astype('float32'))
svm_gpu = cuSVC(class_weight='balanced', C=0.5, max_iter=2000)
svm_gpu.fit(X_gpu, y_gpu)
joblib.dump(svm_gpu, f'{MODEL_DIR_A}/svm.pkl')

# 3c. Naive Bayes
print("Training Naive Bayes...")
X_tr_bin = (X_train_scaled > 0).astype(np.float32)
nb = BernoulliNB(alpha=1.0)
nb.fit(X_tr_bin, y_train)
joblib.dump(nb, f'{MODEL_DIR_A}/nb.pkl')

Training Logistic Regression...
Training cuML SVM for standalone use...
Training Naive Bayes...


['/content/drive/MyDrive/race_rc_project/models/model_a/traditional/nb.pkl']

## 4. Ensemble Construction
We use sklearn's `LinearSVC` here to ensure perfect compatibility with `VotingClassifier` and `CalibratedClassifierCV`.

In [17]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC

print("Building Ensemble...")
# Use sklearn SVM for the ensemble to avoid cuML cloning issues
svm_sk = LinearSVC(class_weight='balanced', C=0.5, max_iter=2000, dual='auto')
svm_cal = CalibratedClassifierCV(svm_sk, cv=3) # Let it handle calibration via 3-fold

ensemble = VotingClassifier(
    estimators=[
        ('lr', lr),
        ('svm', svm_cal),
        ('nb', nb)
    ],
    voting='soft'
)

# Fit on a 20k subset for the ensemble to save time in Colab while maintaining quality
SUBSET = 20000
print(f"Fitting Ensemble on {SUBSET} rows...")
ensemble.fit(X_train_scaled[:SUBSET], y_train[:SUBSET])

joblib.dump(ensemble, f'{MODEL_DIR_A}/ensemble.pkl')
print("Ensemble Achievement Unlocked!")

Building Ensemble...
Fitting Ensemble on 20000 rows...
Ensemble Achievement Unlocked!
